```
# Lab type: review
# Course: ML402 — Reinforcement Learning
# Lesson: Q-Learning
# Task: The implementation below has three bugs. All run without error.
#       For each bug: (1) identify it, (2) explain why it breaks the algorithm,
#       (3) write the corrected version in the fix cell.
```

In [ ]:
import numpy as np
import gymnasium as gym

np.random.seed(42)


## Background

A correct tabular Q-learning implementation on `FrozenLake-v1` should:

1. Use the **off-policy TD update**: `Q[s, a] += α * (r + γ * np.max(Q[s']) - Q[s, a])`
2. Apply **ε-greedy with decay**: ε starts at 1.0 and falls toward a small minimum over training
3. Report a **win rate** by tracking episode outcomes, not reading the last step's reward

Review the code below against these three requirements.

In [ ]:
# --- CODE TO REVIEW ---
# Read carefully before running. What is wrong?

env = gym.make("FrozenLake-v1", is_slippery=True)
n_states  = env.observation_space.n   # 16
n_actions = env.action_space.n        # 4

Q = np.zeros((n_states, n_actions))

alpha   = 0.8
gamma   = 0.95
epsilon = 1.0     # starts at 1.0

for episode in range(5_000):
    state, _ = env.reset()
    done = False

    while not done:
        # ε-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-learning update (??)
        td_error = reward + gamma * Q[next_state, action] - Q[state, action]  # line A
        Q[state, action] += alpha * td_error

        state = next_state

    # epsilon = ... (nothing here — line B)

env.close()

# Report performance
print(f"Final win rate: {reward:.1f}")   # line C


### Bug 1 — Wrong TD target (line A)

**Identify:** Which line contains the bug, and what does it compute instead of the correct Q-learning target?

*(Write your answer here.)*

**Why it breaks the algorithm:** 

*(Write your answer here.)*

In [ ]:
# Fix for Bug 1 — correct Q-learning TD target

env = gym.make("FrozenLake-v1", is_slippery=True)
Q_fix1 = np.zeros((n_states, n_actions))

for episode in range(5_000):
    state, _ = env.reset()
    done = False
    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_fix1[state])
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # FIX: use max over all next actions (off-policy, not on-policy)
        td_error = reward + gamma * np.max(Q_fix1[next_state]) - Q_fix1[state, action]
        Q_fix1[state, action] += alpha * td_error
        state = next_state

env.close()
print("Bug 1 fixed. Q-table sample (state 0):", Q_fix1[0].round(3))


### Bug 2 — ε never decays (line B)

**Identify:** What is missing after the episode loop, and what is the observable symptom at the end of 5 000 episodes?

*(Write your answer here.)*

**Why it breaks the algorithm:**

*(Write your answer here.)*

In [ ]:
# Fix for Bug 2 — add exponential ε decay after each episode

env = gym.make("FrozenLake-v1", is_slippery=True)
Q_fix2 = np.zeros((n_states, n_actions))

epsilon      = 1.0
epsilon_min  = 0.01
epsilon_decay = 0.9990

for episode in range(5_000):
    state, _ = env.reset()
    done = False
    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_fix2[state])
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        td_error = reward + gamma * np.max(Q_fix2[next_state]) - Q_fix2[state, action]
        Q_fix2[state, action] += alpha * td_error
        state = next_state

    # FIX: decay ε after every episode
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

env.close()
print(f"Final ε: {epsilon:.4f}  (should be near {epsilon_min})")


### Bug 3 — Win rate evaluated incorrectly (line C)

**Identify:** What does `reward` hold at the point `print(f"Final win rate: {reward:.1f}")` is executed? What should be printed instead?

*(Write your answer here.)*

**Why it breaks the algorithm:**

*(Write your answer here.)*

In [ ]:
# Fix for Bug 3 — track episode outcomes explicitly

env = gym.make("FrozenLake-v1", is_slippery=True)
Q_fix3 = np.zeros((n_states, n_actions))

epsilon      = 1.0
epsilon_min  = 0.01
epsilon_decay = 0.9990
wins = []  # FIX: collect episode outcomes

for episode in range(5_000):
    state, _ = env.reset()
    done = False
    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_fix3[state])
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        td_error = reward + gamma * np.max(Q_fix3[next_state]) - Q_fix3[state, action]
        Q_fix3[state, action] += alpha * td_error
        state = next_state
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    wins.append(1 if reward == 1.0 else 0)  # FIX: record win/loss per episode

env.close()

# FIX: report win rate over last 200 episodes using the greedy policy
final_win_rate = np.mean(wins[-200:]) * 100
print(f"Win rate (last 200 episodes): {final_win_rate:.1f}%")
for start in range(0, 5000, 1000):
    chunk = wins[start:start+1000]
    print(f"  Episodes {start+1:>5}–{start+1000:<5}  win rate: {np.mean(chunk)*100:.1f}%")


## Summary

> **For each bug, write one sentence on what went wrong and what the correct behaviour should be.**

1.
2.
3.